# 04 - Governance & Maintenance

Governança OPTIMIZE/VACUUM/Lineage

## 1. OPTIMIZE

Toda escrita numa tabela Delta gera arquivos físicos novos - especialmente
após operações incrementais como o MERGE INTO que fizemos em silver.orders.
Com o tempo, isso acumula muitos arquivos pequenos, o que deixa a leitura
mais lenta (o Spark precisa abrir e fechar cada um).

OPTIMIZE compacta esses arquivos pequenos em arquivos maiores, sem alterar
o conteúdo da tabela — só a forma como está organizada fisicamente em disco.

In [0]:
OPTIMIZE olist_project.silver.orders

In [0]:
OPTIMIZE olist_project.gold.fact_orders

### Validação do OPTIMIZE

O resultado de cada OPTIMIZE mostra métricas como numFilesAdded e
numFilesRemoved - confirma quantos arquivos pequenos foram compactados
em arquivos maiores.

## 2. VACUUM

Cada operação no Delta Lake (overwrite, MERGE INTO) mantém os arquivos
antigos por baixo dos panos - é isso que permite o Time Travel. Só que
isso acumula espaço em disco com o tempo.

VACUUM apaga fisicamente os arquivos que não são mais referenciados por
nenhuma versão dentro do período de retenção.

RETAIN 168 HOURS = 7 dias, a retenção padrão e mínima recomendada pelo
Delta Lake. Abaixo disso, o Time Travel de operações concorrentes em
andamento pode quebrar.

**Atenção:** depois de rodar VACUUM, não é mais possível fazer Time
Travel para versões mais antigas que os arquivos removidos. Isso não é
um bug - é o próprio propósito do comando.

⚠️ Atenção real aqui: depois de rodar VACUUM, você perde a capacidade de fazer Time Travel pra versões mais antigas que os arquivos removidos. Isso é importante documentar no README como comportamento esperado - não é bug, é o próprio propósito do comando.

In [0]:
VACUUM olist_project.silver.orders RETAIN 168 HOURS

In [0]:
VACUUM olist_project.gold.fact_orders RETAIN 168 HOURS

## 3. Lineage Graph

Verificado manualmente via Catalog Explorer, não por código:

`Catalog -> olist_project -> gold -> fact_orders -> aba Lineage`

O grafo mostra o fluxo completo:
`bronze.orders + bronze.order_items -> silver.orders + silver.order_items
-> gold.fact_orders`

Print salvo em `screenshots/lineage_fact_orders.png` e referenciado no README.